<a href="https://colab.research.google.com/github/Aditya-sharma112245/music-recommendation-system/blob/main/music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.listdir()

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
df=pd.read_csv('/content/dataset.csv')

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
import pickle

with open("embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

print("✅ Embeddings loaded!")

In [ ]:
df.isnull()

In [ ]:
df = df[["track_name", "artists", "track_genre", "valence"]]

In [ ]:
df.head()

In [ ]:
def f(x):
  if x>0 and x<0.4:
    return "sad"
  elif x>0.4 and x<0.6:
    return "neutral"
  else:
    return "happy"


In [ ]:
df['mood']=df['valence'].apply(f)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df = df[["track_name", "artists", "track_genre", "mood"]]

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
df["track_name"] = df["track_name"].str.lower()
df["artists"] = df["artists"].str.lower()
df["track_genre"] = df["track_genre"].str.lower()
df["artists"] = df["artists"].str.replace(";", ", ")
df["track_name"] = df["track_name"].str.strip()
df["artists"] = df["artists"].str.strip()

In [ ]:
df.head()

In [ ]:
df["text"] = df.apply(
    lambda row: f"Song: {row['track_name']} | Artist: {row['artists']} | Genre: {row['track_genre']} | Mood: {row['mood']}",
    axis=1
)

In [ ]:
df.head()

In [ ]:
!pip install sentence-transformers

In [ ]:
df.size

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
print(df.index[:10])

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
embeddings = model.encode(df["text"].tolist(), show_progress_bar=True)

In [ ]:
print(len(df), embeddings.shape)

In [ ]:
type(embeddings)

In [ ]:
print(embeddings.shape)

In [ ]:
print(embeddings[0])

In [ ]:
query="happy song"
q=model.encode([query])
scores = cosine_similarity(q, embeddings)

In [ ]:
print(scores[:10])

In [ ]:
import urllib.parse

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import urllib.parse

mood_words = {
    "sad": ["sad", "breakup", "heartbroken"],
    "happy": ["happy", "fun", "joy"],
    "party": ["party", "dance", "club"],
    "chill": ["chill", "relax", "calm"]
}

genre_words = ["rock", "pop", "hip-hop", "acoustic", "metal"]


def parse_query(query):
    query = query.lower()

    mood = None
    genre = None

    for key, words in mood_words.items():
        for w in words:
            if w in query:
                mood = key

    for g in genre_words:
        if g in query:
            genre = g

    return mood, genre


def recommend(query, df, embeddings, top_k=5):

    mood, genre = parse_query(query)

    filtered_df = df.copy()


    if mood:
        filtered_df = filtered_df[filtered_df["mood"] == mood]


    if genre:
        filtered_df = filtered_df[
            filtered_df["track_genre"].str.contains(genre, case=False, na=False)
        ]


    if "movie" in query or "film" in query:
        movie_df = df[df["track_genre"].str.contains("film", case=False, na=False)]
        filtered_df = pd.concat([filtered_df, movie_df])


    if len(filtered_df) < 5:
        filtered_df = df


    filtered_df = filtered_df.reset_index()
    filtered_embeddings = embeddings[filtered_df["index"]]


    query_mod = f"{query} song music"
    query_emb = model.encode([query_mod])


    scores = cosine_similarity(query_emb, filtered_embeddings)[0]


    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = filtered_df.iloc[top_indices][["track_name", "artists", "track_genre"]].copy()
    results["score"] = scores[top_indices]


    results["bonus"] = results["track_name"].str.contains(query, case=False, na=False).astype(int)
    results["final_score"] = results["score"] + 0.1 * results["bonus"]

    results = results.sort_values(by="final_score", ascending=False)


    results["youtube"] = results.apply(
        lambda row: "https://www.youtube.com/results?search_query=" +
        urllib.parse.quote(f"{row['track_name']} {row['artists']} lyrics official video"),
        axis=1
    )


    results = results.drop_duplicates(subset="track_name", keep="first")

    return results.head(top_k)

In [ ]:
recommend("funny songs", df, embeddings)



In [ ]:
recommend("jesus", df, embeddings)

In [ ]:
recommend("arijit singh songs", df, embeddings)

In [ ]:
recommend("senorita", df, embeddings)

In [ ]:
recommend("poo", df, embeddings)

In [ ]:
import gradio as gr

def gradio_recommend(query):
    results = recommend(query, df, embeddings)

    if results.empty:
        return "❌ No songs found. Try different query."


    if results["score"].max() < 0.4:
        warning = "⚠️ Results may be less accurate. Try a more specific query.\n\n"
    else:
        warning = ""

    output = warning

    for i, (_, row) in enumerate(results.iterrows(), 1):
        output += f"""
🎵 **{i}. {row['track_name']}**
👤 {row['artists']}
🎧 {row['track_genre']}
⭐ Score: {round(row['score'], 3)}
▶️ [Watch on YouTube]({row['youtube']})

━━━━━━━━━━━━━━━━━━━━
"""

    return output


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🎵 Smart Music Recommender")
    gr.Markdown("Search songs by mood, genre, artist or vibe using AI 🔥")

    with gr.Row():
        with gr.Column():
            query = gr.Textbox(
                label="Search Music 🎧",
                placeholder="Try: sad songs, arijit singh, party music..."
            )

            submit = gr.Button("🔥 Recommend")
            clear = gr.Button("🧹 Clear")

        with gr.Column():
            output = gr.Markdown(label="Recommended Songs 🎶")


    gr.Examples(
        examples=[
            "sad songs",
            "happy party songs",
            "arijit singh",
            "chill night music",
            "kalank movie songs"
        ],
        inputs=query
    )

    submit.click(fn=gradio_recommend, inputs=query, outputs=output)
    clear.click(fn=lambda: "", inputs=None, outputs=output)

demo.launch()

In [ ]:
import pickle


with open("embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

print("✅ Embeddings saved!")